# OPTED Reverse Dictionary - Preprocessing Pipeline

Here we prepare the raw OPTED dictionary file for baseline and model experiments.

The raw file is kept unchanged. Cleaned files are saved separately under `data/processed/`.

## Step 1 - Load the raw dataset

Read the original CSV file. No changes are made to the raw dataset.

In [1]:
from pathlib import Path
import hashlib
import random
import re

import pandas as pd

raw_data_candidates = [
    Path("..") / "data" / "OPTED-Dictionary.csv",
    Path("data") / "OPTED-Dictionary.csv",
]

RAW_DATA_PATH = next(
    (path.resolve() for path in raw_data_candidates if path.exists()),
    None,
)

if RAW_DATA_PATH is None:
    raise FileNotFoundError("Could not find data/OPTED-Dictionary.csv")

ROOT_DIR = RAW_DATA_PATH.parent.parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

WORD_COL = "Word"
COUNT_COL = "Count"
POS_COL = "POS"
DEF_COL = "Definition"

raw_df = pd.read_csv(RAW_DATA_PATH, keep_default_na=False, dtype=str)

print(f"Loaded rows: {len(raw_df):,}")
display(raw_df.head())

Loaded rows: 176,009


,Word,Count,POS,Definition
0,A,1,"""""","""The first letter of the English and of many o..."
1,A,1,"""""","""The name of the sixth tone in the model major..."
2,A,1,"""""","""An adjective commonly called the indefinite ..."
3,A,1,"""""","""In each; to or for each; as """"""""twenty leagu..."
4,A,1,"""prep.""","""In; on; at; by."""


## Step 2 - Validate required columns

Check that the expected OPTED columns are available before continuing.

In [2]:
required_columns = {WORD_COL, COUNT_COL, POS_COL, DEF_COL}
missing_columns = required_columns - set(raw_df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("All required columns are present.")

All required columns are present.


## Step 3 - Clean text fields

Clean extra quotes, repeated spaces, and inconsistent casing.

In [3]:
def normalize_spacing(value):
    text = str(value)
    text = text.replace('""""', '"')
    text = text.strip()
    text = text.strip('"')
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def basic_definition_clean(value):
    text = normalize_spacing(value).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


df = raw_df.copy()

df["word_original"] = df[WORD_COL].map(normalize_spacing)
df["definition_original"] = df[DEF_COL].map(normalize_spacing)
df["pos_original"] = df[POS_COL].map(normalize_spacing)
df["count_original"] = df[COUNT_COL].map(normalize_spacing)

df["word_norm"] = df["word_original"].str.lower()
df["definition_norm"] = df["definition_original"].str.lower()
df["definition_basic_clean"] = df["definition_original"].map(basic_definition_clean)

display(df[["word_original", "word_norm", "definition_original", "definition_basic_clean"]].head())

,word_original,word_norm,definition_original,definition_basic_clean
0,A,a,The first letter of the English and of many ot...,the first letter of the english and of many ot...
1,A,a,The name of the sixth tone in the model major ...,the name of the sixth tone in the model major ...
2,A,a,An adjective commonly called the indefinite ar...,an adjective commonly called the indefinite ar...
3,A,a,"In each; to or for each; as ""twenty leagues a ...",in each to or for each as twenty leagues a day...
4,A,a,In; on; at; by.,in on at by


## Step 4 - Remove unusable rows

Remove rows without a usable word or definition. Spreadsheet artifacts such as `#NAME?` are also removed.

In [4]:
before_rows = len(df)

invalid_word_values = {"", "#name?", "nan", "none"}

df = df[
    ~df["word_norm"].isin(invalid_word_values)
    & df["definition_norm"].ne("")
    & df["definition_basic_clean"].ne("")
].copy()

removed_rows = before_rows - len(df)

print(f"Rows removed because word/definition was blank after cleaning: {removed_rows:,}")
print(f"Rows remaining: {len(df):,}")

Rows removed because word/definition was blank after cleaning: 216
Rows remaining: 175,793


## Step 5 - Remove duplicate word-definition pairs

Remove repeated word-definition pairs. Different definitions for the same word are kept.

In [5]:
before_dedup_rows = len(df)

df = (
    df.drop_duplicates(subset=["word_norm", "definition_norm"])
    .reset_index(drop=True)
    .copy()
)

duplicate_rows_removed = before_dedup_rows - len(df)

print(f"Duplicate word-definition rows removed: {duplicate_rows_removed:,}")
print(f"Rows remaining: {len(df):,}")

Duplicate word-definition rows removed: 239
Rows remaining: 175,554


## Step 6 - Remove stop words and add boundary tokens

Create a stopword-removed definition column and add `<START>` / `<END>` tokens for sequence models.


In [6]:
BASIC_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "by",
    "for", "from", "has", "have", "having", "he", "her", "his", "in",
    "is", "it", "its", "of", "on", "or", "that", "the", "their",
    "them", "they", "this", "to", "was", "were", "which", "who", "with",
}

def remove_stopwords(text):
    tokens = str(text).split()
    return " ".join(token for token in tokens if token not in BASIC_STOPWORDS)


def add_boundary_tokens(text):
    text = str(text).strip()
    return f"<START> {text} <END>" if text else "<START> <END>"


df["definition_no_stopwords"] = df["definition_basic_clean"].map(remove_stopwords)
df["definition_model_input"] = df["definition_basic_clean"].map(add_boundary_tokens)
df["definition_no_stopwords_model_input"] = df["definition_no_stopwords"].map(add_boundary_tokens)

display(
    df[
        [
            "definition_basic_clean",
            "definition_no_stopwords",
            "definition_model_input",
        ]
    ].head()
)


,definition_basic_clean,definition_no_stopwords,definition_model_input
0,the first letter of the english and of many ot...,first letter english many other alphabets capi...,<START> the first letter of the english and of...
1,the name of the sixth tone in the model major ...,name sixth tone model major scale c first tone...,<START> the name of the sixth tone in the mode...
2,an adjective commonly called the indefinite ar...,adjective commonly called indefinite article s...,<START> an adjective commonly called the indef...
3,in each to or for each as twenty leagues a day...,each each twenty leagues day hundred pounds ye...,<START> in each to or for each as twenty leagu...
4,in on at by,,<START> in on at by <END>


## Step 7 - Preserve multiple senses

Keep each word-definition pair as its own sense. This matters because many words have multiple meanings.


In [7]:
def make_entry_id(row):
    key = f"{row['word_norm']}\t{row['definition_norm']}"
    return hashlib.md5(key.encode("utf-8")).hexdigest()[:12]


df = df.sort_values(["word_norm", "definition_norm"]).reset_index(drop=True)
df["sense_number"] = df.groupby("word_norm").cumcount() + 1
df["entry_id"] = df.apply(make_entry_id, axis=1)

display(df[["entry_id", "word_original", "sense_number", "definition_original"]].head(10))

,entry_id,word_original,sense_number,definition_original
0,4a92354c44a6,'Em,1,An obsolete or colloquial contraction of the o...
1,e6a183d531df,'Gainst,1,A contraction of Against.
2,c42603209a5e,'Mongst,1,See Amongst.
3,9fc15aeb2e7a,'Neath,1,An abbreviation of Beneath.
4,dfba73470c2c,'s,1,A contraction for is or (colloquially) for has.
5,7c5da024d47a,'Sblood,1,An abbreviation of God's blood; -- used as an ...
6,73b26104ffe8,'Sdeath,1,An exclamation expressive of impatience or anger.
7,2e97ca3060f9,'Snails,1,God's nails or His nails that is the nails wit...
8,dcd51078682b,'Swounds,1,An exclamation contracted from God's wounds; -...
9,fc9911158cd1,'T is,1,A common contraction of it is.


## Step 8 - Resolve cross-reference definitions

Rows like `See Bovate.` do not explain the word by themselves. Here we try to replace them with the referenced word's real definition.

When the referenced word has more than one definition, the POS tags are compared and the closest match is used. Rows that cannot be matched cleanly are kept out of the baseline-ready files.


In [8]:
CROSS_REFERENCE_PATTERN = re.compile(
    r"^\s*(see|same as|alt\.?\s+of|alternative form of|another form of)\b",
    flags=re.IGNORECASE,
)
CROSS_REFERENCE_PREFIX_PATTERN = re.compile(
    r"^\s*(see|same as|alt\.?\s+of|alternative form of|another form of)\s+",
    flags=re.IGNORECASE,
)

POS_REFERENCE_STOP_TOKENS = {
    "a", "adj", "adv", "conj", "interj", "n", "p", "pl", "prep",
    "pron", "sing", "v", "i", "t", "imp", "ppr", "vb", "obs",
}


def normalize_pos_tokens(pos_text):
    clean_pos = normalize_spacing(pos_text).lower()
    clean_pos = re.sub(r"[^a-z]+", " ", clean_pos)
    return frozenset(token for token in clean_pos.split() if token)


def extract_reference_word(definition_text):
    text = normalize_spacing(definition_text)
    if not CROSS_REFERENCE_PATTERN.search(text):
        return ""

    reference_text = CROSS_REFERENCE_PREFIX_PATTERN.sub("", text, count=1).strip()
    reference_text = re.split(r"[;,(]", reference_text, maxsplit=1)[0]
    reference_text = reference_text.strip(" .:")

    tokens = re.findall(r"[A-Za-z][A-Za-z'-]*\.?", reference_text)
    kept_tokens = []
    for token in tokens:
        clean_token = token.strip(".").lower()
        if kept_tokens and clean_token in POS_REFERENCE_STOP_TOKENS:
            break
        kept_tokens.append(token.strip("."))

    return " ".join(kept_tokens).strip()


def pos_jaccard_score(base_tokens, candidate_tokens):
    if not base_tokens or not candidate_tokens:
        return 0.0
    return len(base_tokens & candidate_tokens) / len(base_tokens | candidate_tokens)


def choose_reference_definition(row, reference_lookup):
    if not row["is_cross_reference_definition"]:
        return pd.Series(
            {
                "cross_reference_resolution_status": "not_cross_reference",
                "cross_reference_target_entry_id": "",
                "cross_reference_resolved_definition": "",
                "cross_reference_pos_score": 0.0,
            }
        )

    target_norm = str(row["cross_reference_target_word"]).lower()
    candidates = reference_lookup.get(target_norm, [])
    if not candidates:
        return pd.Series(
            {
                "cross_reference_resolution_status": "unresolved_missing_reference",
                "cross_reference_target_entry_id": "",
                "cross_reference_resolved_definition": "",
                "cross_reference_pos_score": 0.0,
            }
        )

    if len(candidates) == 1:
        candidate = candidates[0]
        return pd.Series(
            {
                "cross_reference_resolution_status": "resolved_single_reference",
                "cross_reference_target_entry_id": candidate["entry_id"],
                "cross_reference_resolved_definition": candidate["definition_original"],
                "cross_reference_pos_score": pos_jaccard_score(
                    row["pos_token_set"], candidate["pos_token_set"]
                ),
            }
        )

    base_pos_tokens = row["pos_token_set"]
    exact_matches = [
        candidate for candidate in candidates
        if base_pos_tokens and candidate["pos_token_set"] == base_pos_tokens
    ]
    if len(exact_matches) == 1:
        candidate = exact_matches[0]
        return pd.Series(
            {
                "cross_reference_resolution_status": "resolved_exact_pos",
                "cross_reference_target_entry_id": candidate["entry_id"],
                "cross_reference_resolved_definition": candidate["definition_original"],
                "cross_reference_pos_score": 1.0,
            }
        )

    scored_candidates = []
    for candidate in candidates:
        overlap_count = len(base_pos_tokens & candidate["pos_token_set"])
        jaccard_score = pos_jaccard_score(base_pos_tokens, candidate["pos_token_set"])
        scored_candidates.append((overlap_count, jaccard_score, candidate))

    best_overlap = max(score[0] for score in scored_candidates)
    best_jaccard = max(score[1] for score in scored_candidates if score[0] == best_overlap)
    best_candidates = [
        candidate for overlap, score, candidate in scored_candidates
        if overlap == best_overlap and score == best_jaccard
    ]

    if best_overlap > 0 and len(best_candidates) == 1:
        candidate = best_candidates[0]
        return pd.Series(
            {
                "cross_reference_resolution_status": "resolved_max_pos_overlap",
                "cross_reference_target_entry_id": candidate["entry_id"],
                "cross_reference_resolved_definition": candidate["definition_original"],
                "cross_reference_pos_score": best_jaccard,
            }
        )

    return pd.Series(
        {
            "cross_reference_resolution_status": "unresolved_ambiguous_pos",
            "cross_reference_target_entry_id": "",
            "cross_reference_resolved_definition": "",
            "cross_reference_pos_score": best_jaccard,
        }
    )


df["definition_before_cross_reference_resolution"] = df["definition_original"]
df["is_cross_reference_definition"] = df["definition_original"].map(
    lambda text: bool(CROSS_REFERENCE_PATTERN.search(str(text)))
)
df["cross_reference_target_word"] = df["definition_original"].map(extract_reference_word)
df["cross_reference_target_word_norm"] = df["cross_reference_target_word"].str.lower()
df["pos_token_set"] = df["pos_original"].map(normalize_pos_tokens)

reference_source_df = df[~df["is_cross_reference_definition"]].copy()
reference_lookup = {}
for _, candidate_row in reference_source_df.iterrows():
    reference_lookup.setdefault(candidate_row["word_norm"], []).append(
        {
            "entry_id": candidate_row["entry_id"],
            "definition_original": candidate_row["definition_original"],
            "pos_token_set": candidate_row["pos_token_set"],
        }
    )

resolution_df = df.apply(
    lambda row: choose_reference_definition(row, reference_lookup),
    axis=1,
)
df = pd.concat([df, resolution_df], axis=1)

df["is_resolved_cross_reference"] = df["cross_reference_resolution_status"].str.startswith("resolved")
df["is_unresolved_cross_reference"] = (
    df["is_cross_reference_definition"] & ~df["is_resolved_cross_reference"]
)

resolved_mask = df["is_resolved_cross_reference"]
df.loc[resolved_mask, "definition_original"] = df.loc[
    resolved_mask, "cross_reference_resolved_definition"
]

# Rebuild definition columns after resolved rows receive the referenced definition.
df["definition_norm"] = df["definition_original"].str.lower()
df["definition_basic_clean"] = df["definition_original"].map(basic_definition_clean)
df["definition_no_stopwords"] = df["definition_basic_clean"].map(remove_stopwords)
df["definition_model_input"] = df["definition_basic_clean"].map(add_boundary_tokens)
df["definition_no_stopwords_model_input"] = df["definition_no_stopwords"].map(add_boundary_tokens)

cross_reference_resolution_summary = (
    df["cross_reference_resolution_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="rows")
)

sample_words = ["Chinoidine", "Quab", "Oxgang"]
sample_resolution_df = df[
    df["word_original"].isin(sample_words)
][
    [
        "word_original",
        "pos_original",
        "definition_before_cross_reference_resolution",
        "cross_reference_target_word",
        "cross_reference_resolution_status",
        "cross_reference_pos_score",
        "definition_original",
    ]
]

print(f"Cross-reference rows found: {int(df['is_cross_reference_definition'].sum()):,}")
print(f"Resolved cross-reference rows: {int(df['is_resolved_cross_reference'].sum()):,}")
print(f"Unresolved cross-reference rows: {int(df['is_unresolved_cross_reference'].sum()):,}")
display(cross_reference_resolution_summary)
display(sample_resolution_df)


Cross-reference rows found: 9,393
Resolved cross-reference rows: 5,825
Unresolved cross-reference rows: 3,568


,status,rows
0,not_cross_reference,166161
1,resolved_single_reference,5449
2,unresolved_ambiguous_pos,2099
3,unresolved_missing_reference,1469
4,resolved_exact_pos,323
5,resolved_max_pos_overlap,53


,word_original,pos_original,definition_before_cross_reference_resolution,cross_reference_target_word,cross_reference_resolution_status,cross_reference_pos_score,definition_original
26476,Chinoidine,n.,See Quinodine.,Quinodine,unresolved_missing_reference,0.0,See Quinodine.
107757,Oxgang,n.,See Bovate.,Bovate,resolved_single_reference,1.0,An oxgang or as much land as an ox can plow in...
123295,Quab,n.,An unfledged bird; hence something immature or...,,not_cross_reference,0.0,An unfledged bird; hence something immature or...
123296,Quab,v. i.,See Quob v. i.,Quob,resolved_single_reference,1.0,To throb; to quiver.


## Step 9 - Add quality and length features

Add word counts and simple flags for very short or very long definitions.


In [9]:
word_pattern = re.compile(r"\b[\w'-]+\b")

df["definition_word_count"] = df["definition_original"].map(
    lambda text: len(word_pattern.findall(text))
)
df["clean_definition_word_count"] = df["definition_basic_clean"].map(
    lambda text: len(text.split())
)

df["is_short_definition"] = df["definition_word_count"] < 3
df["is_long_definition"] = df["definition_word_count"] > df["definition_word_count"].quantile(0.99)

display(
    df[
        [
            "word_original",
            "definition_original",
            "definition_word_count",
            "clean_definition_word_count",
            "is_short_definition",
            "is_long_definition",
        ]
    ].head()
)

,word_original,definition_original,definition_word_count,clean_definition_word_count,is_short_definition,is_long_definition
0,'Em,An obsolete or colloquial contraction of the o...,11,11,False,False
1,'Gainst,A contraction of Against.,4,4,False,False
2,'Mongst,See Amongst.,2,2,True,False
3,'Neath,An abbreviation of Beneath.,4,4,False,False
4,'s,A contraction for is or (colloquially) for has.,8,8,False,False


## Step 10 - Create word summary files

Save one report for all usable words and another for words that have multiple definitions.


In [10]:
word_summary_source_df = df[~df["is_unresolved_cross_reference"]].copy()

unique_words_df = (
    word_summary_source_df.groupby("word_norm")
    .agg(
        word_original=("word_original", "first"),
        num_entries=("entry_id", "size"),
        num_definitions=("definition_norm", "nunique"),
    )
    .sort_values(["num_definitions", "word_norm"], ascending=[False, True])
    .reset_index()
)

multiple_definition_words_df = unique_words_df[
    unique_words_df["num_definitions"] > 1
].copy()

print(f"Unique words: {len(unique_words_df):,}")
print(f"Words with multiple definitions: {len(multiple_definition_words_df):,}")
display(unique_words_df.head(10))
display(multiple_definition_words_df.head(10))


Unique words: 108,839
Words with multiple definitions: 24,781


,word_norm,word_original,num_entries,num_definitions
0,run,Run,71,71
1,set,Set,59,59
2,light,Light,47,47
3,cast,Cast,45,45
4,round,Round,45,45
5,line,Line,44,44
6,strike,Strike,44,44
7,rise,Rise,42,42
8,fall,Fall,41,41
9,point,Point,41,41


,word_norm,word_original,num_entries,num_definitions
0,run,Run,71,71
1,set,Set,59,59
2,light,Light,47,47
3,cast,Cast,45,45
4,round,Round,45,45
5,line,Line,44,44
6,strike,Strike,44,44
7,rise,Rise,42,42
8,fall,Fall,41,41
9,point,Point,41,41


## Step 11 - Create train, validation, and test splits

Split by connected word groups. If a resolved row says `Y -> See X`, then `Y` and `X` stay in the same train/validation/test split.


In [11]:
RANDOM_SEED = 42
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10

all_words = sorted(df["word_norm"].unique())
parent = {word: word for word in all_words}


def find_group(word):
    while parent[word] != word:
        parent[word] = parent[parent[word]]
        word = parent[word]
    return word


def union_words(left_word, right_word):
    left_root = find_group(left_word)
    right_root = find_group(right_word)
    if left_root != right_root:
        parent[right_root] = left_root


existing_words = set(all_words)
reference_edges_df = df[
    df["is_resolved_cross_reference"]
    & df["cross_reference_target_word_norm"].isin(existing_words)
][["word_norm", "cross_reference_target_word_norm"]].drop_duplicates()

for _, edge in reference_edges_df.iterrows():
    union_words(edge["word_norm"], edge["cross_reference_target_word_norm"])

df["split_group_id"] = df["word_norm"].map(lambda word: find_group(word))

split_groups = pd.Series(df["split_group_id"].unique(), name="split_group_id")
shuffled_groups = split_groups.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

train_end = int(len(shuffled_groups) * TRAIN_RATIO)
valid_end = int(len(shuffled_groups) * (TRAIN_RATIO + VALID_RATIO))

train_groups = set(shuffled_groups.iloc[:train_end])
valid_groups = set(shuffled_groups.iloc[train_end:valid_end])
test_groups = set(shuffled_groups.iloc[valid_end:])


def assign_split(split_group_id):
    if split_group_id in train_groups:
        return "train"
    if split_group_id in valid_groups:
        return "valid"
    return "test"


df["split"] = df["split_group_id"].map(assign_split)

resolved_reference_split_check = df[
    df["is_resolved_cross_reference"]
    & df["cross_reference_target_word_norm"].isin(existing_words)
][
    ["word_norm", "cross_reference_target_word_norm", "split_group_id", "split"]
].copy()

split_summary = (
    df.groupby("split")
    .agg(
        rows=("entry_id", "size"),
        unique_words=("word_norm", "nunique"),
        split_groups=("split_group_id", "nunique"),
        avg_definition_words=("definition_word_count", "mean"),
    )
    .round(2)
    .reset_index()
)

print(f"Reference links used for grouped split: {len(reference_edges_df):,}")
print(f"Split groups created: {df['split_group_id'].nunique():,}")
display(split_summary)


Reference links used for grouped split: 5,819
Split groups created: 105,769


,split,rows,unique_words,split_groups,avg_definition_words
0,test,17826,11138,10577,11.77
1,train,140218,89283,84615,11.57
2,valid,17510,11145,10577,11.23


## Step 12 - Select final columns

Keep the columns needed for modeling, evaluation, and later error analysis.


In [12]:
df["source_entry_id"] = df["entry_id"]
df["augmentation_type"] = "original"

processed_columns = [
    "entry_id",
    "source_entry_id",
    "augmentation_type",
    "split",
    "split_group_id",
    "word_original",
    "word_norm",
    "sense_number",
    "definition_original",
    "definition_before_cross_reference_resolution",
    "definition_norm",
    "definition_basic_clean",
    "definition_no_stopwords",
    "definition_model_input",
    "definition_no_stopwords_model_input",
    "is_cross_reference_definition",
    "is_resolved_cross_reference",
    "is_unresolved_cross_reference",
    "cross_reference_target_word",
    "cross_reference_target_entry_id",
    "cross_reference_resolution_status",
    "cross_reference_pos_score",
    "definition_word_count",
    "clean_definition_word_count",
    "is_short_definition",
    "is_long_definition",
    "count_original",
    "pos_original",
]

processed_df = df[processed_columns].copy()
baseline_ready_df = processed_df[
    ~processed_df["is_unresolved_cross_reference"]
].reset_index(drop=True)

cross_reference_summary = pd.DataFrame(
    [
        {
            "total_rows": len(processed_df),
            "cross_reference_rows": int(processed_df["is_cross_reference_definition"].sum()),
            "resolved_cross_reference_rows": int(processed_df["is_resolved_cross_reference"].sum()),
            "unresolved_cross_reference_rows": int(processed_df["is_unresolved_cross_reference"].sum()),
            "baseline_ready_rows": len(baseline_ready_df),
        }
    ]
)

split_summary = (
    baseline_ready_df.groupby("split")
    .agg(
        rows=("entry_id", "size"),
        unique_words=("word_norm", "nunique"),
        split_groups=("split_group_id", "nunique"),
        avg_definition_words=("definition_word_count", "mean"),
    )
    .round(2)
    .reset_index()
)

display(processed_df.head())
display(cross_reference_summary)
print(f"Final processed rows: {len(processed_df):,}")
print(f"Baseline-ready rows: {len(baseline_ready_df):,}")


,entry_id,source_entry_id,augmentation_type,split,split_group_id,word_original,word_norm,sense_number,definition_original,definition_before_cross_reference_resolution,...,cross_reference_target_word,cross_reference_target_entry_id,cross_reference_resolution_status,cross_reference_pos_score,definition_word_count,clean_definition_word_count,is_short_definition,is_long_definition,count_original,pos_original
0,4a92354c44a6,4a92354c44a6,original,train,'em,'Em,'em,1,An obsolete or colloquial contraction of the o...,An obsolete or colloquial contraction of the o...,...,,,not_cross_reference,0.0,11,11,False,False,3,
1,e6a183d531df,e6a183d531df,original,train,'gainst,'Gainst,'gainst,1,A contraction of Against.,A contraction of Against.,...,,,not_cross_reference,0.0,4,4,False,False,7,prep.
2,c42603209a5e,c42603209a5e,original,test,'mongst,'Mongst,'mongst,1,See Amongst.,See Amongst.,...,Amongst,,unresolved_ambiguous_pos,1.0,2,2,True,False,7,prep.
3,9fc15aeb2e7a,9fc15aeb2e7a,original,train,'neath,'Neath,'neath,1,An abbreviation of Beneath.,An abbreviation of Beneath.,...,,,not_cross_reference,0.0,4,4,False,False,6,prep. & adv.
4,dfba73470c2c,dfba73470c2c,original,train,'s,'s,'s,1,A contraction for is or (colloquially) for has.,A contraction for is or (colloquially) for has.,...,,,not_cross_reference,0.0,8,8,False,False,2,


,total_rows,cross_reference_rows,resolved_cross_reference_rows,unresolved_cross_reference_rows,baseline_ready_rows
0,175554,9393,5825,3568,171986


Final processed rows: 175,554
Baseline-ready rows: 171,986


## Step 13 - Basic training augmentation

Create a small augmented training file using synonym replacement, random deletion, and random insertion. Validation and test data are not augmented.


In [13]:
AUGMENT_FRACTION = 0.1
MAX_AUGMENT_SOURCE_ROWS = 10000
AUGMENTATION_RANDOM_SEED = 42

BASIC_SYNONYMS = {
    "large": ["big"],
    "small": ["little"],
    "quick": ["fast"],
    "begin": ["start"],
    "end": ["finish"],
    "make": ["create"],
    "remove": ["delete"],
    "join": ["connect"],
    "part": ["portion"],
    "kind": ["type"],
    "place": ["location"],
    "person": ["individual"],
    "thing": ["object"],
    "use": ["employ"],
}

def synonym_replacement(tokens, rng, max_replacements=1):
    new_tokens = list(tokens)
    candidate_indices = [
        index for index, token in enumerate(new_tokens) if token in BASIC_SYNONYMS
    ]
    rng.shuffle(candidate_indices)

    replacements = 0
    for index in candidate_indices[:max_replacements]:
        new_tokens[index] = rng.choice(BASIC_SYNONYMS[new_tokens[index]])
        replacements += 1

    return new_tokens if replacements else None


def random_deletion(tokens, rng, deletion_probability=0.10):
    if len(tokens) <= 3:
        return None

    new_tokens = [
        token for token in tokens if rng.random() > deletion_probability
    ]

    if len(new_tokens) == len(tokens):
        remove_index = rng.randrange(len(tokens))
        new_tokens = tokens[:remove_index] + tokens[remove_index + 1:]

    return new_tokens if new_tokens else None


def random_insertion(tokens, rng, max_insertions=1):
    if not tokens:
        return None

    new_tokens = list(tokens)
    synonym_source_tokens = [token for token in tokens if token in BASIC_SYNONYMS]

    for _ in range(max_insertions):
        if synonym_source_tokens:
            source_token = rng.choice(synonym_source_tokens)
            inserted_token = rng.choice(BASIC_SYNONYMS[source_token])
        else:
            inserted_token = rng.choice(tokens)

        insert_index = rng.randrange(len(new_tokens) + 1)
        new_tokens.insert(insert_index, inserted_token)

    return new_tokens


def build_augmented_row(row, augmented_tokens, augmentation_type):
    augmented_text = " ".join(augmented_tokens)
    augmented_row = row.to_dict()
    augmented_row["entry_id"] = f"{row['entry_id']}_{augmentation_type}"
    augmented_row["source_entry_id"] = row["entry_id"]
    augmented_row["augmentation_type"] = augmentation_type
    augmented_row["definition_original"] = augmented_text
    augmented_row["definition_norm"] = augmented_text
    augmented_row["definition_basic_clean"] = augmented_text
    augmented_row["definition_no_stopwords"] = remove_stopwords(augmented_text)
    augmented_row["definition_model_input"] = add_boundary_tokens(augmented_text)
    augmented_row["definition_no_stopwords_model_input"] = add_boundary_tokens(
        augmented_row["definition_no_stopwords"]
    )
    augmented_row["definition_before_cross_reference_resolution"] = augmented_text
    augmented_row["is_cross_reference_definition"] = False
    augmented_row["is_resolved_cross_reference"] = False
    augmented_row["is_unresolved_cross_reference"] = False
    augmented_row["cross_reference_target_word"] = ""
    augmented_row["cross_reference_target_entry_id"] = ""
    augmented_row["cross_reference_resolution_status"] = "not_cross_reference"
    augmented_row["cross_reference_pos_score"] = 0.0
    augmented_row["definition_word_count"] = len(augmented_tokens)
    augmented_row["clean_definition_word_count"] = len(augmented_tokens)
    augmented_row["is_short_definition"] = len(augmented_tokens) < 3
    augmented_row["is_long_definition"] = False
    return augmented_row


rng = random.Random(AUGMENTATION_RANDOM_SEED)
train_original_df = baseline_ready_df[baseline_ready_df["split"] == "train"].copy()
augmentation_candidate_df = train_original_df.copy()
augment_source_count = min(
    MAX_AUGMENT_SOURCE_ROWS,
    max(1, int(len(augmentation_candidate_df) * AUGMENT_FRACTION)),
)

augmentation_source_df = augmentation_candidate_df.sample(
    n=augment_source_count,
    random_state=AUGMENTATION_RANDOM_SEED,
)

augmented_rows = []
seen_augmented_examples = set()
augmentation_methods = {
    "synonym_replacement": synonym_replacement,
    "random_deletion": random_deletion,
    "random_insertion": random_insertion,
}

for _, row in augmentation_source_df.iterrows():
    tokens = str(row["definition_basic_clean"]).split()
    for augmentation_type, augmentation_function in augmentation_methods.items():
        augmented_tokens = augmentation_function(tokens, rng)
        if not augmented_tokens or augmented_tokens == tokens:
            continue

        augmented_text = " ".join(augmented_tokens)
        example_key = (row["entry_id"], augmentation_type, augmented_text)
        if example_key in seen_augmented_examples:
            continue

        seen_augmented_examples.add(example_key)
        augmented_rows.append(
            build_augmented_row(row, augmented_tokens, augmentation_type)
        )

augmented_df = pd.DataFrame(augmented_rows, columns=processed_columns)
train_augmented_basic_df = pd.concat(
    [train_original_df, augmented_df],
    ignore_index=True,
)

augmentation_summary = (
    augmented_df["augmentation_type"]
    .value_counts()
    .rename_axis("augmentation_type")
    .reset_index(name="rows")
)

print(f"Training rows before augmentation: {len(train_original_df):,}")
print(f"Training rows eligible for augmentation: {len(augmentation_candidate_df):,}")
print(f"Source rows sampled for augmentation: {augment_source_count:,}")
print(f"Augmented rows created: {len(augmented_df):,}")
print(f"Training rows after augmentation: {len(train_augmented_basic_df):,}")
display(augmentation_summary)
display(augmented_df[["word_original", "augmentation_type", "definition_basic_clean"]].head(10))


Training rows before augmentation: 137,362
Training rows eligible for augmentation: 137,362
Source rows sampled for augmentation: 10,000
Augmented rows created: 18,942
Training rows after augmentation: 156,304


,augmentation_type,rows
0,random_insertion,10000
1,random_deletion,7838
2,synonym_replacement,1104


,word_original,augmentation_type,definition_basic_clean
0,Sabadilla,random_deletion,a liliaceous plant schoenocaulon officinale al...
1,Sabadilla,random_insertion,a mexican liliaceous contain plant schoenocaul...
2,Mount,synonym_replacement,that upon which a person or object is mounted
3,Mount,random_deletion,upon which a person or thing is mounted
4,Mount,random_insertion,that object upon which a person or thing is mo...
5,Incito-motor,random_deletion,to motion to that action which in the case of ...
6,Incito-motor,random_insertion,inciting to motion applied to that action whic...
7,Wended,random_insertion,wend of wend
8,Inadequation,random_deletion,want of correspondence
9,Inadequation,random_insertion,want of of exact correspondence


## Step 14 - Save processed files

Save one cleaned dataset, one set of splits, vocabulary reports, and the basic augmented training file.


In [14]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

baseline_ready_df.to_csv(PROCESSED_DIR / "opted_preprocessed.csv", index=False)

for split_name, split_df in baseline_ready_df.groupby("split"):
    split_df.to_csv(PROCESSED_DIR / f"opted_{split_name}.csv", index=False)

train_augmented_basic_df.to_csv(
    PROCESSED_DIR / "opted_train_augmented_basic.csv",
    index=False,
)
unique_words_df.to_csv(PROCESSED_DIR / "unique_words.csv", index=False)
multiple_definition_words_df.to_csv(
    PROCESSED_DIR / "multiple_definition_words.csv",
    index=False,
)
split_summary.to_csv(PROCESSED_DIR / "split_summary.csv", index=False)
augmentation_summary.to_csv(PROCESSED_DIR / "augmentation_summary.csv", index=False)

print(f"Saved processed files to: {PROCESSED_DIR}")
print("Created:")
print("- opted_preprocessed.csv")
print("- opted_train.csv")
print("- opted_train_augmented_basic.csv")
print("- opted_valid.csv")
print("- opted_test.csv")
print("- unique_words.csv")
print("- multiple_definition_words.csv")
print("- split_summary.csv")
print("- augmentation_summary.csv")

Saved processed files to: D:\Git\MLCapstone_ReverseDict\data\processed
Created:
- opted_preprocessed.csv
- opted_train.csv
- opted_train_augmented_basic.csv
- opted_valid.csv
- opted_test.csv
- unique_words.csv
- multiple_definition_words.csv
- split_summary.csv
- augmentation_summary.csv


## Preprocessing summary

In this notebook, the raw English OPTED dictionary data was cleaned and prepared for baseline modeling.

The main preprocessing steps were:

- Loaded the raw OPTED dataset and checked that the required columns were present.
- Cleaned extra quotes, spaces, word text, POS labels, and definition text.
- Removed rows with blank words or unusable definitions.
- Removed duplicate word-definition pairs while keeping different meanings of the same word.
- Created stopword-removed definitions for simpler text experiments.
- Added `<START>` and `<END>` tokens for later sequence-based models.
- Resolved simple `See ...` dictionary references when the referenced word could be matched clearly.
- Used POS overlap to choose the closest referenced definition when more than one option was available.
- Kept unresolved or ambiguous cross-reference rows out of the baseline-ready files.
- Kept each word-definition pair as a separate sense.
- Created reports for unique words and words with multiple definitions.
- Added simple definition length flags to help identify very short or very long definitions.
- Split the data into train, validation, and test sets using connected word groups, so linked reference words stay together.
- Created a basic train-only augmented dataset using synonym replacement, random deletion, and random insertion.
- Saved the cleaned CSV files under `data/processed/` for baseline modeling.